<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 04. PCA: Reducción de Dimensionalidad sin Dolor
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/06%20-%20Feature%20Engineering/Para%20Dummies/04_PCA_Feature_Engineering_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## PCA: encontrar la "mejor sombra" de tus datos 🔦

Imagina que sostienes una tetera de cerámica frente a una linterna y proyectas su sombra en la pared. Si iluminas desde un ángulo raro, la sombra es una mancha confusa. Pero si encuentras el ángulo correcto, la sombra 2D deja ver claramente el pico, el mango y la tapa — casi toda la información de la tetera 3D, en solo dos dimensiones.

El **Análisis de Componentes Principales (PCA)** hace exactamente eso, pero con números: si tienes una tabla con muchas columnas (a veces decenas), busca el "ángulo" perfecto para proyectarlas en unas pocas columnas nuevas — llamadas **componentes principales** — que conservan la mayor cantidad posible de la información original.

---
## 1. ¿Por qué funciona esto? Los "ejes naturales" de los datos 📏

Piensa en dos medidas de una persona: **estatura** y **envergadura de brazos** (la distancia entre las puntas de los dedos con los brazos extendidos). Estas dos variables casi siempre van de la mano: alguien más alto casi siempre tiene los brazos más largos también.

Si las graficas una contra otra, los puntos no forman una nube redonda y desordenada — forman una especie de "cigarro" alargado en diagonal. Dentro de esa nube hay dos direcciones especiales:
1. **La dirección más larga del cigarro:** captura el "tamaño general" de la persona (alto + brazos largos vs. bajo + brazos cortos). A esto lo llamamos **PC1**.
2. **La dirección perpendicular, más corta:** captura lo que sobra — por ejemplo, personas con brazos relativamente más largos o más cortos que lo esperado para su estatura. A esto lo llamamos **PC2**.

PCA simplemente encuentra esas direcciones matemáticamente, para cualquier número de columnas — no solo dos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Simulamos estatura y envergadura de brazos: muy correlacionadas
np.random.seed(7)
estatura = np.random.normal(170, 10, 200)
envergadura = estatura * 1.02 + np.random.normal(0, 4, 200)  # muy parecida a la estatura, con algo de ruido

df_cuerpo = pd.DataFrame({'Estatura_cm': estatura, 'Envergadura_cm': envergadura})

# PCA siempre se aplica sobre datos estandarizados (media 0, desviación 1)
X_scaled = StandardScaler().fit_transform(df_cuerpo)

pca = PCA(n_components=2)
componentes = pca.fit_transform(X_scaled)

print('Varianza explicada por cada componente:')
print(f'  PC1: {pca.explained_variance_ratio_[0]*100:.1f}%')
print(f'  PC2: {pca.explained_variance_ratio_[1]*100:.1f}%')

### 🤔 ¿Qué acaba de pasar?

- `StandardScaler()` deja ambas columnas en la misma escala (media 0, desviación 1). Esto es obligatorio antes de PCA: si no lo hicieras, la variable con números más grandes dominaría el resultado solo por su tamaño, no porque sea más importante.
- `PCA(n_components=2)` calculó las dos direcciones nuevas (PC1 y PC2) que vimos en la analogía del "cigarro".
- El resultado muestra que **PC1 concentra casi toda la varianza** (probablemente más del 95%), justo porque estatura y envergadura están casi perfectamente correlacionadas: casi toda la información "cabe" en una sola dirección — el "tamaño general" del cuerpo.

---
## 2. Las "cargas" (*loadings*): ¿de qué está hecho cada componente? ⚖️

Cada componente principal es simplemente una **receta**: una suma ponderada de las columnas originales. A esos "pesos de la receta" se les llama **cargas (*loadings*)**. Mirarlos te dice qué significa cada componente en términos que un humano entiende.

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=df_cuerpo.columns
)
loadings.round(3)

### 🤔 ¿Qué acaba de pasar?

- En `PC1`, tanto `Estatura_cm` como `Envergadura_cm` tienen pesos positivos y muy parecidos: eso confirma que `PC1` es un "índice de tamaño general" — sube cuando ambas variables suben juntas.
- En `PC2`, los pesos suelen tener signos distintos (uno positivo, uno negativo): ese componente captura el "desbalance" — personas con envergadura más larga o corta de lo esperado para su estatura.
- No necesitas memorizar los números exactos; lo importante es la idea: **el signo y el tamaño de las cargas te cuentan la historia de cada componente**.

---
## 3. ¿Para qué sirve PCA en la práctica? 🛠️

| Uso | Analogía |
|---|---|
| **Reducir dimensiones** | Guardar solo las 2-3 "sombras" más informativas en vez de 50 columnas originales |
| **Detectar valores atípicos** | Un dato raro suele "sobresalir" en los componentes de menor varianza |
| **Reducir ruido** | La señal común queda en los primeros componentes; el ruido aleatorio, en los últimos |
| **Quitar correlación entre variables** | Los componentes son, por construcción, completamente independientes entre sí |

Vamos a verlo con un dataset real: 13 medidas químicas de vinos de tres variedades distintas. En vez de graficar 13 columnas (¡imposible!), usamos PCA para "aplanarlas" a solo 2 y así poder verlas en un plano.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine()
X = wine.data
y = wine.target

# 1. Estandarizar siempre antes de PCA
X_scaled = StandardScaler().fit_transform(X)

# 2. Reducir de 13 columnas a solo 2 componentes
pca_wine = PCA(n_components=2)
X_pca = pca_wine.fit_transform(X_scaled)

var_exp = pca_wine.explained_variance_ratio_
print(f'Varianza capturada por PC1 + PC2: {sum(var_exp)*100:.1f}% (de las 13 variables originales)')

plt.figure(figsize=(7, 5))
for target, color, label in zip([0, 1, 2], ['#e11d48', '#2563eb', '#16a34a'], wine.target_names):
    plt.scatter(X_pca[y == target, 0], X_pca[y == target, 1], label=label, color=color, alpha=0.7, s=45)
plt.xlabel(f'PC1 ({var_exp[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({var_exp[1]*100:.1f}%)')
plt.title('13 características químicas del vino, proyectadas a 2D')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 🤔 ¿Qué acaba de pasar?

- Pasamos de **13 columnas químicas** a solo **2 componentes**, y aun así se logran distinguir con bastante claridad las tres variedades de vino en el gráfico — la mayoría de la información "esencial" sobrevivió a la compresión.
- Esto es justo la idea de la sombra de la tetera: no vemos cada detalle químico por separado, pero sí vemos la forma general que diferencia a cada grupo.
- Si necesitaras usar estas variables en un modelo con muchas columnas correlacionadas entre sí, alimentar el modelo con `PC1` y `PC2` en lugar de las 13 originales suele simplificar el entrenamiento sin perder mucha señal.

---
### ✅ Autocomprobación Rápida

**Pregunta:** ¿Por qué es obligatorio estandarizar (`StandardScaler`) las columnas antes de aplicar PCA?

<details>
<summary>💡 Ver respuesta</summary>

Porque PCA busca las direcciones de mayor **varianza**, y la varianza depende de la escala de los números. Si una columna está en pesos colombianos (números en millones) y otra en años (números pequeños), la de pesos "dominará" el resultado solo por tener números más grandes — no porque sea más importante. Al estandarizar, todas las columnas quedan en pie de igualdad (media 0, desviación 1), y PCA puede comparar su variación de forma justa.

</details>

---
## 4. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| PCA | Encuentra los "ángulos" que mejor resumen tus columnas numéricas en menos dimensiones |
| Componente principal (PC1, PC2...) | Una combinación ponderada de las columnas originales |
| Cargas (*loadings*) | Los "pesos" de esa combinación; su signo y tamaño explican qué mide cada componente |
| Varianza explicada | Qué porcentaje de la información original conserva cada componente |
| Estandarizar antes de PCA | Obligatorio: evita que una columna domine solo por tener números más grandes |

➡️ **Siguiente paso:** en el cuaderno [05 - Selección de Características e Información Mutua (Para Dummies)](05_Seleccion_Caracteristicas_y_Mutual_Information_Dummies.ipynb) aprenderás a elegir cuáles variables vale la pena conservar — y cuáles solo agregan ruido.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
